In [ ]:
import os
import re
import gc
import shutil
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from diffusers import DiffusionPipeline


INPUT_ROOT = Path("input/small_parts")

if not INPUT_ROOT.exists():
    raise FileNotFoundError(f"Input folder not found: {INPUT_ROOT}")

WORK_DIR = Path("outputs")
OUTPUT_ROOT = WORK_DIR / "test_set_depth_normal_png_background"
OUTPUT_ZIP = WORK_DIR / "test_set_depth_normal_png_background.zip"

IMAGE_EXTENSIONS = [".png"]

PRESERVE_ORIGINAL_EXTENSION = True
CLEAR_OUTPUT_ROOT_ON_START = True
SKIP_EXISTING = True
STOP_ON_ERROR = True

WHITE_BACKGROUND_THRESHOLD = 245


DEPTH_MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
NORMAL_MODEL_ID = "GonzaloMG/stable-diffusion-e2e-ft-normals"
NORMAL_CUSTOM_PIPELINE = "GonzaloMG/marigold-e2e-ft-normals"
NORMAL_PROCESSING_RES = 768

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NORMAL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True


if CLEAR_OUTPUT_ROOT_ON_START and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def natural_key(path):
    text = str(path)
    return [
        int(s) if s.isdigit() else s.lower()
        for s in re.split(r"(\d+)", text)
    ]


def find_subfolders(root):
    folders = [
        p for p in Path(root).iterdir()
        if p.is_dir()
    ]

    folders = sorted(folders, key=natural_key)

    if len(folders) == 0:
        raise RuntimeError(f"No subfolders found inside: {root}")

    return folders


def find_image_files(folder):
    files = [
        p for p in Path(folder).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    return sorted(files, key=natural_key)


def output_name_from_input(image_path):
    if PRESERVE_ORIGINAL_EXTENSION:
        return image_path.name

    return image_path.stem + ".png"


def normalize01(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    mn = float(np.min(x))
    mx = float(np.max(x))

    if mx > mn:
        return (x - mn) / (mx - mn)

    return np.zeros_like(x, dtype=np.float32)


def save_gray(path, x):
    x8 = np.clip(x * 255.0, 0, 255).astype(np.uint8)
    Image.fromarray(x8).save(path)


def save_rgb(path, x):
    x8 = np.clip(x, 0, 255).astype(np.uint8)
    Image.fromarray(x8).save(path)


def load_png_with_mask(path):
    img = Image.open(path).convert("RGBA")
    rgba = np.array(img)

    rgb = rgba[..., :3].astype(np.uint8)
    alpha = rgba[..., 3].astype(np.uint8)

    if np.min(alpha) < 255:
        mask = alpha
    else:
        white_pixels = np.all(rgb >= WHITE_BACKGROUND_THRESHOLD, axis=-1)
        mask = np.where(white_pixels, 0, 255).astype(np.uint8)

    m = mask.astype(np.float32)[..., None] / 255.0

    rgb_white = (
        rgb.astype(np.float32) * m
        + 255.0 * (1.0 - m)
    )

    rgb_white = np.clip(rgb_white, 0, 255).astype(np.uint8)

    return rgb_white, mask


def apply_depth_white_background(depth, mask):
    m = mask.astype(np.float32) / 255.0
    depth = depth * m + 1.0 * (1.0 - m)
    return np.clip(depth, 0.0, 1.0)


def apply_normal_white_background(normals, mask):
    m = mask.astype(np.float32) / 255.0

    normals = np.clip(normals, -1.0, 1.0)
    normals = normals / (np.linalg.norm(normals, axis=-1, keepdims=True) + 1e-8)

    normal_rgb = (normals + 1.0) * 0.5
    normal_rgb = normal_rgb * m[..., None] + 1.0 * (1.0 - m[..., None])

    return np.clip(normal_rgb, 0.0, 1.0)


def normal_output_to_hwc_array(x):
    if torch.is_tensor(x):
        x = x.detach().float().cpu().numpy()
    else:
        x = np.asarray(x).astype(np.float32)

    if x.ndim == 4:
        x = x[0]

    if x.ndim == 3 and x.shape[0] == 3 and x.shape[-1] != 3:
        x = np.transpose(x, (1, 2, 0))

    if x.ndim != 3 or x.shape[-1] != 3:
        raise RuntimeError(f"Unexpected normal output shape: {x.shape}")

    if x.max() > 1.5:
        x = x / 255.0

    if x.min() >= -1e-6 and x.max() <= 1.05:
        x = x * 2.0 - 1.0

    return x.astype(np.float32)


print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Device:", DEVICE)

print("Loading depth model...")

depth_processor = AutoImageProcessor.from_pretrained(
    DEPTH_MODEL_ID,
    use_fast=False,
)

depth_model = AutoModelForDepthEstimation.from_pretrained(
    DEPTH_MODEL_ID,
).to(DEVICE)

depth_model.eval()

print("Loading normal model...")

normal_pipe = DiffusionPipeline.from_pretrained(
    NORMAL_MODEL_ID,
    custom_pipeline=NORMAL_CUSTOM_PIPELINE,
    torch_dtype=NORMAL_DTYPE,
).to(DEVICE)

if DEVICE == "cuda":
    normal_pipe.enable_attention_slicing()

print("Models loaded.")


def predict_depth(rgb):
    h, w = rgb.shape[:2]

    rgb_np = np.ascontiguousarray(rgb.astype(np.uint8))

    try:
        inputs = depth_processor(
            images=rgb_np,
            return_tensors="pt",
        )
    except TypeError:
        inputs = depth_processor(
            images=[rgb_np],
            return_tensors="pt",
        )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        outputs = depth_model(**inputs)

        predicted_depth = outputs.predicted_depth

        prediction = torch.nn.functional.interpolate(
            predicted_depth.unsqueeze(1),
            size=(h, w),
            mode="bicubic",
            align_corners=False,
        )

    depth = prediction.squeeze().detach().cpu().numpy().astype(np.float32)
    depth = normalize01(depth)

    depth = 1.0 - depth

    return depth


def predict_normals(rgb):
    h, w = rgb.shape[:2]

    rgb_np = np.ascontiguousarray(rgb.astype(np.uint8))

    image_tensor = torch.from_numpy(rgb_np).permute(2, 0, 1).unsqueeze(0)
    image_tensor = image_tensor.to(device=DEVICE, dtype=NORMAL_DTYPE) / 255.0

    with torch.inference_mode():
        out = normal_pipe(
            image_tensor,
            processing_resolution=NORMAL_PROCESSING_RES,
            match_input_resolution=True,
            output_type="np",
        )

    if hasattr(out, "prediction"):
        normals = normal_output_to_hwc_array(out.prediction)

    elif hasattr(out, "images"):
        normals = normal_output_to_hwc_array(out.images)

    elif isinstance(out, dict):
        if "prediction" in out:
            normals = normal_output_to_hwc_array(out["prediction"])
        elif "images" in out:
            normals = normal_output_to_hwc_array(out["images"])
        else:
            raise RuntimeError("Normal prediction output dictionary was not recognized.")

    elif isinstance(out, (tuple, list)) and len(out) > 0:
        normals = normal_output_to_hwc_array(out[0])

    else:
        raise RuntimeError("Normal prediction output was not recognized.")

    if normals.shape[:2] != (h, w):
        normals = cv2.resize(
            normals,
            (w, h),
            interpolation=cv2.INTER_LINEAR,
        )

    normals = np.clip(normals, -1.0, 1.0)
    normals = normals / (np.linalg.norm(normals, axis=-1, keepdims=True) + 1e-8)

    return normals


subfolders = find_subfolders(INPUT_ROOT)

total_frames = 0

for folder in subfolders:
    total_frames += len(find_image_files(folder))

print(f"Found {len(subfolders)} folders.")
print(f"Found {total_frames} total PNG frames.")

processed_total = 0

for folder_index, folder in enumerate(subfolders, start=1):
    image_files = find_image_files(folder)

    if len(image_files) == 0:
        print(f"Skipping empty folder: {folder.name}")
        continue

    folder_output = OUTPUT_ROOT / folder.name
    depth_dir = folder_output / "depth"
    normal_dir = folder_output / "normal"

    depth_dir.mkdir(parents=True, exist_ok=True)
    normal_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nFolder {folder_index}/{len(subfolders)}: {folder.name}")
    print(f"Frames found: {len(image_files)}")

    for frame_index, image_path in enumerate(image_files, start=1):
        out_name = output_name_from_input(image_path)

        depth_path = depth_dir / out_name
        normal_path = normal_dir / out_name

        if SKIP_EXISTING and depth_path.exists() and normal_path.exists():
            processed_total += 1
            continue

        try:
            rgb_white, mask = load_png_with_mask(image_path)

            depth_raw = predict_depth(rgb_white)
            depth = apply_depth_white_background(
                depth=depth_raw,
                mask=mask,
            )

            normal_vec = predict_normals(rgb_white)
            normal_rgb = apply_normal_white_background(
                normals=normal_vec,
                mask=mask,
            )

            save_gray(depth_path, depth)
            save_rgb(normal_path, normal_rgb * 255.0)

            processed_total += 1

            del rgb_white, mask, depth_raw, depth, normal_vec, normal_rgb

        except Exception as e:
            message = f"Error processing {image_path}: {e}"

            if STOP_ON_ERROR:
                raise RuntimeError(message)

            print(message)

        if frame_index % 25 == 0 or frame_index == len(image_files):
            print(
                f"{folder.name}: {frame_index}/{len(image_files)} frames checked | "
                f"Total processed: {processed_total}/{total_frames}"
            )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()


shutil.make_archive(
    base_name=str(OUTPUT_ZIP.with_suffix("")),
    format="zip",
    root_dir=OUTPUT_ROOT,
)

print("\nDone.")
print("Output folder:", OUTPUT_ROOT)
print("Output zip:", OUTPUT_ZIP)
